# Week 3 Day 2 — Types of Data & Data Cleaning
**DSF-PT15 | Moringa School | Student Notebook**

---

## The Story

Every piece of data in a dataset has a type. That type determines what you can do with it.
Get the type right and your analysis works. Get it wrong and you either crash — or worse,
you get a number that looks correct but means nothing.

Today we work with a real dataset: **Financial Inclusion in Africa** — survey data collected
across Kenya, Uganda, Tanzania and Rwanda asking whether people have a bank account and what
factors influence that. Every column is a different data type. We will classify them, clean them,
load the CSV, and calculate our first summary statistics.

**Dataset:** `data/financial_inclusion_africa.csv`  
**Source:** Zindi Financial Inclusion in Africa competition (open dataset)

---

## How to Use This Notebook
- Run cells from top to bottom — order matters
- Every `# TODO` is where you write code
- Every cell has a comment above telling you what to do
- Never leave a TODO blank — write pseudocode first if you get stuck
- `Shift + Enter` runs a cell and moves to the next one

---

## Session Map
- **Stage 1 — Warmup:** What do we already know?
- **Stage 2 — Core Concept:** Data types + categorical vs quantitative
- **Stage 3 — Building the Example:** Load and clean the real dataset
- **Stage 4 — Extend It:** Statistical thinking on our clean data
- **Stage 5 — Your Turn:** Apply everything to a health clinic scenario


---
##  Warmup

Python has 4 core data types. You have used all of them in Module 1.
Run the cells below — just observe. No code to write yet.

In [1]:
# Run this cell and read the output — what type is each variable?

name    = "Amina Wanjiru"   # str  — text
age     = 28               # int  — whole number
balance = 4750.50          # float — decimal number
active  = True             # bool — True or False

print(type(name),    name)
print(type(age),     age)
print(type(balance), balance)
print(type(active),  active)

<class 'str'> Amina Wanjiru
<class 'int'> 28
<class 'float'> 4750.5
<class 'bool'> True


In [2]:
# Run this cell — what happens when you use the wrong type?
# This demonstrates why types matter before we look at any dataset

try:
    result = name + 100   # What do you think this will do?
except TypeError as e:
    print("TypeError:", e)

# This works fine — right types
total = age + 2
print("Age in 2 years:", total)

TypeError: can only concatenate str (not "int") to str
Age in 2 years: 30


**Quick check — answer in your head before moving on:**

Why did the first operation fail but the second one worked?

---
## Stage 2 — Core Concept: What Kind of Data Is This?

Before looking at code, ask these three questions about any column:

**1. Is it a label or a number?**
- A **label** tells you what category something belongs to → `str` (usually)
- A **number** tells you how much of something → `int` or `float`

**2. If it's a label — does it have order?**
- **No order** → Nominal (e.g. country, gender)
- **Has order** → Ordinal (e.g. education: None < Primary < Secondary < Tertiary)
- **Only two options** → Binary (e.g. Yes/No → True/False)

**3. If it's a number — is it countable or measurable?**
- **Countable whole numbers** → Discrete → `int`
- **Any value in a range** → Continuous → `float`

In [3]:
# Run this cell — see what a single row from our dataset looks like when first loaded
# Notice: EVERYTHING is a string. That is what happens when Python reads a CSV.

sample_row = {
    "country":              "Kenya",
    "year":                 "2018",
    "uniqueid":             "uniqueid_1",
    "location_type":        "Rural",
    "cellphone_access":     "Yes",
    "household_size":       "3",
    "age_of_respondent":    "34",
    "gender_of_respondent": "Female",
    "education_level":      "Secondary Education",
    "job_type":             "Self employed",
    "bank_account":         "Yes"
}

for col, val in sample_row.items():
    print(f"{col:30} {type(val).__name__:8}  {val}")

country                        str       Kenya
year                           str       2018
uniqueid                       str       uniqueid_1
location_type                  str       Rural
cellphone_access               str       Yes
household_size                 str       3
age_of_respondent              str       34
gender_of_respondent           str       Female
education_level                str       Secondary Education
job_type                       str       Self employed
bank_account                   str       Yes


**Look at the output above.** Every column shows `str`.

Our job is to convert each one to what it actually is.
Fill in the table mentally before we start coding:

| Column | Should be | Why |
|---|---|---|
| `country` | str | It's a label — Kenya/Uganda etc. |
| `year` | int | It's a count — a whole number |
| `uniqueid` | str | It's an identifier — never calculate with it |
| `cellphone_access` | bool | Yes/No → True/False |
| `household_size` | int | Count of people — whole number |
| `age_of_respondent` | int | Age in years — whole number |
| `education_level` | str (ordinal) | Has order, but leave as str for now |
| `bank_account` | bool | Yes/No → True/False (target column) |

---
## Stage 3 — Building the Example: Load and Clean the Dataset

In [4]:
# Load the dataset using csv.DictReader
# Each row becomes a dictionary with column names as keys

import csv

# TODO: open the file and load it into a list called data
with open(r"C:\Users\STRATH\OneDrive - Strathmore University\Desktop\MORINGA\Module 2\dsf-pt15-student-materials\Financial_inclusion_dataset.csv", encoding="utf-8") as f:
    data = list(csv.DictReader(f))

# How many rows did we load?
print(f"Total rows: {len(data)}")

# What does a single row look like?
print("\nFirst row:")
for col, val in data[0].items():
    print(f"  {col:30} {val}")

Total rows: 23524

First row:
  country                        Kenya
  year                           2018
  uniqueid                       uniqueid_1
  bank_account                   Yes
  location_type                  Rural
  cellphone_access               Yes
  household_size                 3
  age_of_respondent              24
  gender_of_respondent           Female
  relationship_with_head         Spouse
  marital_status                 Married/Living together
  education_level                Secondary education
  job_type                       Self employed


In [5]:
# Filter to Kenya only — more relatable, smaller to work with in class

# TODO: create kenya_data — a list of only the rows where country == "Kenya"
# Hint: use a list comprehension — [row for row in data if row[___] == ___]
kenya_data = [row for row in data if row["country"] == "Kenya"]

print(f"Kenya rows:    {len(kenya_data)}")
print(f"Full dataset:  {len(data)}")
print(f"Kenya is {len(kenya_data)/len(data)*100:.1f}% of the full dataset")

Kenya rows:    6068
Full dataset:  23524
Kenya is 25.8% of the full dataset


### 3a — Cleaning String Columns

String cleaning always happens first — before any type conversion.
The key methods: `.strip()`, `.lower()`, `.title()`, `.replace()`

In [6]:
# Run this cell — see why .strip() matters
# Spaces are invisible but break equality checks

dirty = "Self employed"
clean = dirty.strip()

print(f"Before: '{dirty}'")
print(f"After:  '{clean}'")
print(f"Equal without strip? {dirty == 'Self employed'}")
print(f"Equal after strip?   {clean == 'Self employed'}")

Before: 'Self employed'
After:  'Self employed'
Equal without strip? True
Equal after strip?   True


In [7]:
# Clean all string columns in the Kenya dataset
# .strip() removes any leading or trailing whitespace from every text column

string_cols = [
    "country", "uniqueid", "location_type", "cellphone_access",
    "gender_of_respondent", "relationship_with_head",
    "marital_status", "education_level", "job_type", "bank_account"
]

# TODO: loop through kenya_data and apply .strip() to every string column
# Hint:
# for row in kenya_data:
#     for col in string_cols:
#         row[col] = row[col].___
for row in kenya_data:
    for col in string_cols:
        row[col] = row[col].strip()

# Verify — what unique values are in bank_account after cleaning?
bank_values = set(row["bank_account"] for row in kenya_data)
print("Unique values in bank_account:", bank_values)

Unique values in bank_account: {'No', 'Yes'}


In [8]:
# Check for casing inconsistencies in education_level
# Run this and look at the output — do you see any issues?

edu_values = sorted(set(row["education_level"] for row in kenya_data))
print("Education levels found:")
for v in edu_values:
    print(f"  '{v}'")

Education levels found:
  'No formal education'
  'Other/Dont know/RTA'
  'Primary education'
  'Secondary education'
  'Tertiary education'
  'Vocational/Specialised training'


### 3b — Type Conversion

Rule: **if you will do arithmetic with it, convert it. If it is a label, leave it as str.**

In [9]:
# Convert numeric columns from string to their correct Python types
# year, household_size and age_of_respondent are all integers in reality

for row in kenya_data:
    # TODO: convert year to int
    row["year"] = int(row["year"])

    # TODO: convert household_size to int
    row["household_size"] = int(row["household_size"])

    # TODO: convert age_of_respondent to int
    row["age_of_respondent"] = int(row["age_of_respondent"])

# Verify the conversion worked
sample = kenya_data[0]
print(f"year type:              {type(sample['year']).__name__}     value: {sample['year']}")
print(f"household_size type:    {type(sample['household_size']).__name__}     value: {sample['household_size']}")
print(f"age_of_respondent type: {type(sample['age_of_respondent']).__name__}     value: {sample['age_of_respondent']}")

year type:              int     value: 2018
household_size type:    int     value: 3
age_of_respondent type: int     value: 24


In [10]:
# Convert the binary Yes/No columns to booleans
# The pattern: row["column"] = (row["column"] == "Yes")
# This gives True when the value is "Yes" and False when it is anything else

for row in kenya_data:
    # TODO: convert cellphone_access — Yes → True, No → False
    row["cellphone_access"] = (row["cellphone_access"] == "Yes")

    # TODO: convert bank_account — Yes → True, No → False
    row["bank_account"] = (row["bank_account"] == "Yes")

# Verify the types
print(f"cellphone_access type: {type(kenya_data[0]['cellphone_access']).__name__}")
print(f"bank_account type:     {type(kenya_data[0]['bank_account']).__name__}")

# TODO: count how many people in kenya_data have a bank account
# Hint: sum(1 for row in kenya_data if row["bank_account"])
has_account = sum(1 for row in kenya_data if row["bank_account"])
pct = has_account / len(kenya_data) * 100
print(f"\n{has_account} out of {len(kenya_data)} Kenyans have a bank account ({pct:.1f}%)")

cellphone_access type: bool
bank_account type:     bool

1521 out of 6068 Kenyans have a bank account (25.1%)


---
## Stage 4 — Extend It: Statistical Thinking

We now have clean numeric data. Ask the three questions:
- **What is the typical value?** → mean, median, or mode
- **How spread out is it?** → min, max
- **Are there outliers pulling the numbers?** → compare mean vs median

In [11]:
# Calculate mean age — and compare it to the median
# Age is roughly normally distributed — mean is valid here

import numpy as np

# TODO: extract all ages from kenya_data into a list called ages
# Hint: [row["age_of_respondent"] for row in kenya_data]
ages = [row["age_of_respondent"] for row in kenya_data]

# Convert to numpy array
ages_array = np.array(ages)

# TODO: print the mean, median, min and max age
# Hint: ages_array.mean(), np.median(ages_array), ages_array.min(), ages_array.max()
print(f"Mean age:   {ages_array.mean():.1f}")
print(f"Median age: {np.median(ages_array):.1f}")
print(f"Min age:    {ages_array.min()}")
print(f"Max age:    {ages_array.max()}")

Mean age:   39.6
Median age: 35.0
Min age:    16
Max age:    95


In [12]:
# Find the most common job type — this is the MODE
# We use mode for categorical data — mean and median don't work on text

from collections import Counter

# TODO: create a Counter of job types across kenya_data
# Hint: Counter(row["job_type"] for row in kenya_data)
job_counts = Counter(row["job_type"] for row in kenya_data)

# TODO: print the top 3 most common job types
# Hint: job_counts.most_common(3)
print("Top 3 job types in Kenya:")
for job, count in job_counts.most_common(3):
    pct = count / len(kenya_data) * 100
    print(f"  {job:35} {count:5} respondents ({pct:.1f}%)")

Top 3 job types in Kenya:
  Farming and Fishing                  1609 respondents (26.5%)
  Informally employed                  1419 respondents (23.4%)
  Remittance Dependent                 1188 respondents (19.6%)


In [ ]:
# Compare mean vs median for household_size
# A few large households may pull the mean upward — is median better here?

sizes = np.array([row["household_size"] for row in kenya_data])

print(f"Household size — Mean:   {sizes.mean():.2f}")
print(f"Household size — Median: {np.median(sizes):.1f}")
print(f"Household size — Max:    {sizes.max()}")
print()

# TODO: write one sentence explaining which is the better summary statistic here and why
# Replace the string below with your answer
answer_variable  = "The median is a better summary statistic here because a few large households may skew the mean upward."
print(answer_variable)

Household size — Mean:   3.99
Household size — Median: 4.0
Household size — Max:    21

your_answer The median is a better summary statistic here because a few large households may skew the mean upward.


---
## Stage 5 — Your Turn

**Same skills, different data. Do not copy from Stage 3 — think through each step.**

**Scenario:** You work at a Nairobi health clinic. You have been given patient visit records.
Your task:
1. Clean the string columns
2. Convert numeric and boolean columns to the right types
3. Calculate one summary statistic per numeric column
4. Answer: which statistic is most appropriate for `num_visits` and why?

The data is loaded below — you write all the cleaning code.

In [14]:
# The clinic dataset — loaded for you, ready to clean

clinic_data = [
    {"patient_id": "P001", "age": "34",  "gender": "  Female ", "county": "Nairobi  ",  "diagnosis": "Malaria",   "weight_kg": "58.5",  "num_visits": "3",  "on_insurance": "Yes"},
    {"patient_id": "P002", "age": "56",  "gender": "Male",     "county": "Mombasa",    "diagnosis": "Diabetes",  "weight_kg": "82.0",  "num_visits": "12", "on_insurance": "No"},
    {"patient_id": "P003", "age": "22",  "gender": " female",  "county": "  Kisumu ",  "diagnosis": "Malaria",   "weight_kg": "49.0",  "num_visits": "1",  "on_insurance": "Yes"},
    {"patient_id": "P004", "age": "41",  "gender": "MALE",     "county": "Nakuru",     "diagnosis": "Typhoid",   "weight_kg": "75.5",  "num_visits": "2",  "on_insurance": "No"},
    {"patient_id": "P005", "age": "29",  "gender": "Female",   "county": "Nairobi",    "diagnosis": "Flu",       "weight_kg": "61.0",  "num_visits": "1",  "on_insurance": "Yes"},
    {"patient_id": "P006", "age": "67",  "gender": "male ",    "county": "Eldoret",    "diagnosis": "Diabetes",  "weight_kg": "70.0",  "num_visits": "24", "on_insurance": "Yes"},
    {"patient_id": "P007", "age": "18",  "gender": "Female",   "county": "nairobi",    "diagnosis": "Malaria",   "weight_kg": "52.5",  "num_visits": "1",  "on_insurance": "No"},
    {"patient_id": "P008", "age": "45",  "gender": "Male",     "county": "Mombasa  ",  "diagnosis": "Typhoid",   "weight_kg": "88.0",  "num_visits": "4",  "on_insurance": "No"},
]

print(f"Clinic records: {len(clinic_data)}")
print("\nFirst record (raw):")
for col, val in clinic_data[0].items():
    print(f"  {col:15} '{val}'")

Clinic records: 8

First record (raw):
  patient_id      'P001'
  age             '34'
  gender          '  Female '
  county          'Nairobi  '
  diagnosis       'Malaria'
  weight_kg       '58.5'
  num_visits      '3'
  on_insurance    'Yes'


In [15]:
# Step 1 — Clean string columns: strip whitespace + fix casing
# Columns to clean: gender, county
# Use .strip() to remove spaces, .title() to fix casing

for row in clinic_data:
    # TODO: clean gender — strip whitespace and fix casing
    row["gender"] = row["gender"].strip().title()


    # TODO: clean county — strip whitespace and fix casing
    row["county"] = row["county"].strip().title()

# Verify
genders = sorted(set(row["gender"] for row in clinic_data))
counties = sorted(set(row["county"] for row in clinic_data))
print("Genders after cleaning:", genders)
print("Counties after cleaning:", counties)

Genders after cleaning: ['Female', 'Male']
Counties after cleaning: ['Eldoret', 'Kisumu', 'Mombasa', 'Nairobi', 'Nakuru']


In [16]:
# Step 2 — Convert to the correct types
# age → int, weight_kg → float, num_visits → int, on_insurance → bool

for row in clinic_data:
    # TODO: convert age to int
    row["age"] = int(row["age"])


    # TODO: convert weight_kg to float (it has decimals)
    row["weight_kg"] = float(row["weight_kg"])

    # TODO: convert num_visits to int
    row["num_visits"] = int(row["num_visits"])

    # TODO: convert on_insurance to bool — Yes → True, No → False
    row["on_insurance"] = (row["on_insurance"] == "Yes")

# Verify types
r = clinic_data[0]
print(f"age type:          {type(r['age']).__name__}")
print(f"weight_kg type:    {type(r['weight_kg']).__name__}")
print(f"num_visits type:   {type(r['num_visits']).__name__}")
print(f"on_insurance type: {type(r['on_insurance']).__name__}")

age type:          int
weight_kg type:    float
num_visits type:   int
on_insurance type: bool


In [17]:
# Step 3 — Calculate summary statistics for each numeric column
# Use numpy. For each column: calculate both mean AND median, then decide which to use.

import numpy as np

# TODO: extract ages, weights, and visit counts as numpy arrays
ages_c    = np.array([row["age"] for row in clinic_data])
weights_c = np.array([row["weight_kg"] for row in clinic_data])
visits_c  = np.array([row["num_visits"] for row in clinic_data])

print("--- Age ---")
# TODO: print mean and median for age
print(f"  Mean:   {ages_c.mean():.1f}")
print(f"  Median: {np.median(ages_c):.1f}")

print("\n--- Weight (kg) ---")
# TODO: print mean and median for weight
print(f"  Mean:   {weights_c.mean():.1f}")
print(f"  Median: {np.median(weights_c):.1f}")

print("\n--- Number of Visits ---")
# TODO: print mean and median for num_visits
print(f"  Mean:   {visits_c.mean():.1f}")
print(f"  Median: {np.median(visits_c):.1f}")
print(f"  Max:    {visits_c.max()}")

--- Age ---
  Mean:   39.0
  Median: 37.5

--- Weight (kg) ---
  Mean:   67.1
  Median: 65.5

--- Number of Visits ---
  Mean:   6.0
  Median: 2.5
  Max:    24


In [18]:
# Step 4 — Reflection question
# Look at the mean vs median for num_visits.
# Which is the better summary statistic for this column?
# What does the max tell you?

# TODO: replace the string below with your answer (1-2 sentences)
your_reflection = "The median is a better summary statistic for num_visits because the data is skewed by a very high value (24 visits), which pulls the mean upward. The max shows there is an outlier, meaning a few patients visit the clinic much more frequently than most."
print(your_reflection)

The median is a better summary statistic for num_visits because the data is skewed by a very high value (24 visits), which pulls the mean upward. The max shows there is an outlier, meaning a few patients visit the clinic much more frequently than most.


---
## Extension — Early Finishers

If you finish Stage 5, try one of these:

1. In the clinic data, what percentage of patients have insurance? Write the calculation.
2. Find the most common diagnosis in the clinic data using `Counter`.
3. In the Kenya financial inclusion data, compare the mean age of people who have a bank account
   vs those who do not. What do you notice?

In [19]:
# Extension space — pick one challenge above and work on it here

# TODO: your extension code here
# Hint for challenge 3:
# with_account    = [row["age_of_respondent"] for row in kenya_data if row["bank_account"]]
# without_account = [row["age_of_respondent"] for row in kenya_data if not row["bank_account"]]

---
## Session Recap

| Concept | What it means | Example from today |
|---|---|---|
| `str` | Text — labels, names, categories | `country`, `job_type`, `gender` |
| `int` | Whole numbers — countable | `age`, `household_size`, `year` |
| `float` | Decimal numbers — measurable | `weight_kg` |
| `bool` | True or False | `bank_account`, `on_insurance` |
| Nominal | Labels with no order | `country`, `job_type` |
| Ordinal | Labels with order | `education_level` |
| Discrete | Countable whole numbers | `household_size`, `num_visits` |
| Continuous | Any value in a range | `weight_kg` |
| `.strip()` | Remove whitespace | `"  Kenya  "` → `"Kenya"` |
| `.title()` | Fix casing | `"nairobi"` → `"Nairobi"` |
| `int()` | Convert to integer | `"34"` → `34` |
| `float()` | Convert to decimal | `"58.5"` → `58.5` |
| bool conversion | Yes/No → True/False | `row["bank_account"] == "Yes"` |
| Mean | Best for normal data | Average age |
| Median | Best for skewed data | Typical number of visits |
| Mode | Best for categorical data | Most common job type |

---
## Before Monday
- [ ] Canvas: Types of Data (video + reading)
- [ ] Canvas: Technical Lesson — Text in Structured Tabular Data
- [ ] Canvas: Technical Lesson — Numeric and Boolean Data Types
- [ ] Canvas: Technical Lesson — CSV (45 min — hands-on)
- [ ] Canvas: Introduction to JSONs + What is a JSON?
- [ ] Canvas: Data Collection and Sampling Techniques (reading)
- [ ] Canvas: Practice — Types of Data (vet clinic dataset)
- [ ] Canvas: Quiz — Types of Data and Collecting Data
